# Linear 线性层

线性层是神经网络中最基础的模块之一。它做的事情很简单：把输入向量通过一组可学习的权重和偏置，映射到另一个向量空间中。

在数学上，线性层通常写作：

$$
y = Wx + b
$$

不过在实际代码里，我们更常使用 batch 形式，并且输入通常按“行向量”组织。因此本 notebook 中的实现采用的是：

$$
Y = XW + b
$$

其中：

$$
X \in \mathbb{R}^{B \times d_{in}}, \quad
W \in \mathbb{R}^{d_{in} \times d_{out}}, \quad
b \in \mathbb{R}^{d_{out}}, \quad
Y \in \mathbb{R}^{B \times d_{out}}
$$

这里的 $B$ 是 batch size，$d_{in}$ 是输入维度，$d_{out}$ 是输出维度。偏置 $b$ 会在 batch 维度上自动广播，相当于给每一个样本都加上同一组偏置。

严格来说，带有偏置项的 $XW + b$ 是一个仿射变换（affine transformation），但在深度学习里通常仍然把它叫作 linear layer。

## 为什么只堆叠线性层不够

如果两个线性层之间没有非线性激活函数，那么多层线性层叠加之后，本质上仍然可以合并成一个线性层。例如：

$$
h = xW_1 + b_1
$$

$$
y = hW_2 + b_2
$$

代入可得：

$$
y = (xW_1 + b_1)W_2 + b_2 = x(W_1W_2) + (b_1W_2 + b_2)
$$

也就是说，两层线性层可以等价成一层新的线性层：

$$
y = xW^{\prime} + b^{\prime}
$$

其中：

$$
W^{\prime} = W_1W_2, \quad b^{\prime} = b_1W_2 + b_2
$$

因此，单纯增加线性层的层数并不能提高模型表达复杂非线性函数的能力。现实中的图像、文本、语音等问题大多是高度非线性的，所以线性层通常会和非线性激活函数一起使用：

$$
y = f(XW + b)
$$

早期的多层感知机（MLP）就是不断重复“线性层 + 激活函数”的结构。

## 输入输出 shape

在实际应用中，线性层通常只作用在最后一个维度上。

如果输入是普通 batch：

$$
X \in \mathbb{R}^{B \times d_{in}}
$$

那么输出是：

$$
Y \in \mathbb{R}^{B \times d_{out}}
$$

如果输入是序列数据，例如文本 embedding：

$$
X \in \mathbb{R}^{B \times T \times d_{in}}
$$

其中 $T$ 是 sequence length，那么线性层会对每个 token 的最后一维做同样的变换，输出为：

$$
Y \in \mathbb{R}^{B \times T \times d_{out}}
$$

可以直观理解为：对于每一个样本、每一个 token，线性层都会把原来的 $d_{in}$ 维特征重新组合成 $d_{out}$ 维特征。

## 参数学习

线性层中真正需要学习的参数是 $W$ 和 $b$。在反向传播中，如果有：

$$
Y = XW + b
$$

设损失函数对输出的梯度为：

$$
G = \frac{\partial L}{\partial Y}
$$

那么有：

$$
\frac{\partial L}{\partial W} = X^T G
$$

$$
\frac{\partial L}{\partial b} = \sum_{i=1}^{B} G_i
$$

$$
\frac{\partial L}{\partial X} = GW^T
$$

这也是线性层能够通过梯度下降不断更新参数的原因。

本 notebook 下面的代码实现了一个非常简化的 Linear 层，用来帮助理解前向传播中的 shape 变化和矩阵乘法过程。实际工程中还会更关注参数初始化方式、数值稳定性、自动求导和模块注册等问题。


In [3]:
# Linear 层实现
import numpy as np

class Linear:
    def __init__(self,
                 input_dim=None, 
                 output_dim=1):
        # 初始化权重和偏置
        self.weights = None
        self.bias = None
        self.input_dim = input_dim
        self.output_dim = output_dim

    def forward(self, x):
        # x 的 shape 为 (batch_size, input_dim) 或 （batch_size, seq_len, input_dim）
        # 初始化权重和偏置
        if self.weights is None:
            input_dim = self.input_dim
            output_dim = self.output_dim
            self.weights = np.random.randn(input_dim, output_dim)
            self.bias = np.random.randn(output_dim)
        # 计算线性变换
        y = x @ self.weights + self.bias
        return y

In [4]:
# 一个例子
x = np.random.randn(5, 10)  # 假设输入是一个 batch_size 为 5，input_dim 为 10 的矩阵
linear_layer = Linear(10, 3)  # 创建一个 Linear 层，输入维度为 10，输出维度为 3
output = linear_layer.forward(x)  # 前向传播
print("输入 shape:", x.shape)
print("输出 shape:", output.shape)  # 输出的 shape 应该是 (5, 3)

# 如果 x 的 shape 是更常见的 (batch_size, seq_len, input_dim)，例如：
x_seq = np.random.randn(5, 7, 10)  # 假设输入是一个 batch_size 为 5，seq_len 为 7，input_dim 为 10 的矩阵
output_seq = linear_layer.forward(x_seq)  # 前向传播
print("输入 shape:", x_seq.shape)
print("输出 shape:", output_seq.shape)  # 输出的 shape 应该是 (5, 7, 3)

输入 shape: (5, 10)
输出 shape: (5, 3)
输入 shape: (5, 7, 10)
输出 shape: (5, 7, 3)
